In [1]:
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_gait import *
from src.plot_data_gait import *
from src.train_LSTMGait import train_model, predict_trial, evaluate_events, summarize

In [2]:
base_path = '../datasets/GAIT_signals/dataset/data'

In [3]:
lr = 0.001
dropout = 0.5
cnn_channels = [64, 128, 128]
kernel_size = 7
batch_size = 128
hidden_size = 256
window_size = 200
stride = 50
process = "free_raw"
sensors = ["LB"]

# CNN + BiLSTM

In [4]:
X, y, subjects = load_dataset_gait(base_path, process=process, sensors=sensors, window_size=window_size, stride=stride)
X = X.astype(np.float32)
y = y.astype(np.int64)

model_name = "cnn_bilstm_LB_only"

ckpt_path = f"../checkpoints/tolerance_{model_name}"

Trials descartados: 10 ['HS_2_1', 'KOA_17_3', 'KOA_17_4', 'PD_6_4', 'PD_6_5', 'RIL_24_6', 'RIL_24_7', 'RIL_24_9', 'RIL_31_12', 'RIL_42_2'] | sujetos con datos: 260


In [5]:
trained_model, best_val_f1, test_f1, df_val_f1, df_test_f1 = train_model(
    X, y, subjects, model_name="cnnbilstm", norm="window", 
    epochs=200, lr=lr, patience=20, dropout=dropout, 
    cnn_channels=cnn_channels, kernel_size=kernel_size, 
    batch_size=batch_size, hidden_size=hidden_size, out_dir=ckpt_path, plot=True)


Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 68. Best F1: 0.9214

Val F1: 0.9214  |  Test F1: 0.9037
Checkpoint saved → ../checkpoints/tolerance_cnn_bilstm_LB_only/model.pt


In [6]:
print(f"Validation F1 score: {best_val_f1:.4f}")
print(f"Test F1 score: {test_f1:.4f}")

Validation F1 score: 0.9214
Test F1 score: 0.9037


## Event F1 with tolerance

In [ ]:
ck = pd.read_json(ckpt_path + "/summary.json")

results, by_cohort = [], {}
for subj in ck["test_subjects"]:
    _, trials = load_patient(base_path, subj)
    for t in trials:
        try:
            y_pred, y_true, _ = predict_trial(base_path, t, subj, "cnnbilstm", process="free_raw", sensors=["LB"], window_size=200, stride=50, out_dir=ckpt_path)
        except ValueError:
            continue                                         
        r = evaluate_events(y_true, y_pred)
        results.append(r)
        by_cohort.setdefault(subj.split("_")[0], []).append(r)
        
summarize(results)                                          

   model_name    norm    val_f1   test_f1  n_train_subjects  n_val_subjects  \
0   cnnbilstm  window  0.921381  0.903691               166              42   
1   cnnbilstm  window  0.921381  0.903691               166              42   
2   cnnbilstm  window  0.921381  0.903691               166              42   
3   cnnbilstm  window  0.921381  0.903691               166              42   
4   cnnbilstm  window  0.921381  0.903691               166              42   
5   cnnbilstm  window  0.921381  0.903691               166              42   
6   cnnbilstm  window  0.921381  0.903691               166              42   
7   cnnbilstm  window  0.921381  0.903691               166              42   
8   cnnbilstm  window  0.921381  0.903691               166              42   
9   cnnbilstm  window  0.921381  0.903691               166              42   
10  cnnbilstm  window  0.921381  0.903691               166              42   
11  cnnbilstm  window  0.921381  0.903691           

# LSTM

In [8]:
model_name = "lstm_LB_only"

ckpt_path = f"../checkpoints/baseline_{model_name}"

In [9]:
trained_model, best_val_f1, test_f1, df_val_f1, df_test_f1 = train_model(
    X, y, subjects, model_name="lstm", norm="window", 
    epochs=200, lr=lr, patience=20, dropout=dropout, 
    cnn_channels=cnn_channels, kernel_size=kernel_size, 
    batch_size=batch_size, hidden_size=hidden_size, out_dir=ckpt_path, plot=True)


Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 52. Best F1: 0.8758

Val F1: 0.8758  |  Test F1: 0.8536
Checkpoint saved → ../checkpoints/baseline_lstm_LB_only/model.pt


In [10]:
print(f"Validation F1 score: {best_val_f1:.4f}")
print(f"Test F1 score: {test_f1:.4f}")

Validation F1 score: 0.8758
Test F1 score: 0.8536


## Event F1 with tolerance

In [ ]:
ck = pd.read_json(ckpt_path + "/summary.json")

results, by_cohort = [], {}
for subj in ck["test_subjects"]:
    _, trials = load_patient(base_path, subj)
    for t in trials:
        try:
            y_pred, y_true, _ = predict_trial(base_path, t, subj, "lstm", process="free_raw", sensors=["LB"], window_size=200, stride=50, out_dir=ckpt_path)
        except ValueError:
            continue                                         
        r = evaluate_events(y_true, y_pred)
        results.append(r)
        by_cohort.setdefault(subj.split("_")[0], []).append(r)
        
summarize(results)                                          

   model_name    norm    val_f1   test_f1  n_train_subjects  n_val_subjects  \
0        lstm  window  0.875753  0.853596               166              42   
1        lstm  window  0.875753  0.853596               166              42   
2        lstm  window  0.875753  0.853596               166              42   
3        lstm  window  0.875753  0.853596               166              42   
4        lstm  window  0.875753  0.853596               166              42   
5        lstm  window  0.875753  0.853596               166              42   
6        lstm  window  0.875753  0.853596               166              42   
7        lstm  window  0.875753  0.853596               166              42   
8        lstm  window  0.875753  0.853596               166              42   
9        lstm  window  0.875753  0.853596               166              42   
10       lstm  window  0.875753  0.853596               166              42   
11       lstm  window  0.875753  0.853596           

TO: F1=0.858 P=0.780 R=0.954 | error mediano 20 ms [IQR 10-40]
HS: F1=0.856 P=0.778 R=0.952 | error mediano 20 ms [IQR 10-40]


# BiLSTM

In [12]:
model_name = "bilstm_LB_only"

ckpt_path = f"../checkpoints/baseline_{model_name}"

In [13]:
trained_model, best_val_f1, test_f1, df_val_f1, df_test_f1 = train_model(
    X, y, subjects, model_name="bilstm", norm="window", 
    epochs=200, lr=lr, patience=20, dropout=dropout, 
    cnn_channels=cnn_channels, kernel_size=kernel_size, 
    batch_size=batch_size, hidden_size=hidden_size, out_dir=ckpt_path, plot=True)


Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Early stopping in epoch 40. Best F1: 0.9150

Val F1: 0.9150  |  Test F1: 0.8940
Checkpoint saved → ../checkpoints/baseline_bilstm_LB_only/model.pt


In [14]:
print(f"Validation F1 score: {best_val_f1:.4f}")
print(f"Test F1 score: {test_f1:.4f}")

Validation F1 score: 0.9150
Test F1 score: 0.8940


## Event F1 with tolerance

In [ ]:
ck = pd.read_json(ckpt_path + "/summary.json")

results, by_cohort = [], {}
for subj in ck["test_subjects"]:
    _, trials = load_patient(base_path, subj)
    for t in trials:
        try:
            y_pred, y_true, _ = predict_trial(base_path, t, subj, "bilstm", process="free_raw", sensors=["LB"], window_size=200, stride=50, out_dir=ckpt_path)
        except ValueError:
            continue                                         
        r = evaluate_events(y_true, y_pred)
        results.append(r)
        by_cohort.setdefault(subj.split("_")[0], []).append(r)
        
summarize(results)                                          

   model_name    norm    val_f1   test_f1  n_train_subjects  n_val_subjects  \
0      bilstm  window  0.915049  0.894002               166              42   
1      bilstm  window  0.915049  0.894002               166              42   
2      bilstm  window  0.915049  0.894002               166              42   
3      bilstm  window  0.915049  0.894002               166              42   
4      bilstm  window  0.915049  0.894002               166              42   
5      bilstm  window  0.915049  0.894002               166              42   
6      bilstm  window  0.915049  0.894002               166              42   
7      bilstm  window  0.915049  0.894002               166              42   
8      bilstm  window  0.915049  0.894002               166              42   
9      bilstm  window  0.915049  0.894002               166              42   
10     bilstm  window  0.915049  0.894002               166              42   
11     bilstm  window  0.915049  0.894002           

TO: F1=0.940 P=0.922 R=0.960 | error mediano 10 ms [IQR 10-30]
HS: F1=0.938 P=0.919 R=0.957 | error mediano 20 ms [IQR 10-30]


# CNN

In [16]:
model_name = "CNN_LB_only"

ckpt_path = f"../checkpoints/baseline_{model_name}"

In [17]:
trained_model, best_val_f1, test_f1, df_val_f1, df_test_f1 = train_model(
    X, y, subjects, model_name="cnn", norm="window", 
    epochs=200, lr=lr, patience=20, dropout=dropout, 
    cnn_channels=cnn_channels, kernel_size=kernel_size, 
    batch_size=batch_size, hidden_size=hidden_size, out_dir=ckpt_path, plot=True)


Subject split  →  train+val: 208 subjects  |  test: 52 subjects
Window split   →  train+val: 58845  |  test: 16995
               →  train: 166 subjects (45819 windows)  |  val: 42 subjects (13026 windows)


Training:   2%|▎         | 5/200 [00:30<19:35,  6.03s/it, Val F1=0.860, LR=0.001000]

Early stopping in epoch 41. Best F1: 0.8727

Val F1: 0.8727  |  Test F1: 0.8534
Checkpoint saved → ../checkpoints/baseline_CNN_LB_only/model.pt


In [18]:
print(f"Validation F1 score: {best_val_f1:.4f}")
print(f"Test F1 score: {test_f1:.4f}")

Validation F1 score: 0.8727
Test F1 score: 0.8534


## Event F1 with tolerance

In [ ]:
ck = pd.read_json(ckpt_path + "/summary.json")

results, by_cohort = [], {}
for subj in ck["test_subjects"]:
    _, trials = load_patient(base_path, subj)
    for t in trials:
        try:
            y_pred, y_true, _ = predict_trial(base_path, t, subj, "cnn", process="free_raw", sensors=["LB"], window_size=200, stride=50, out_dir=ckpt_path)
        except ValueError:
            continue                                         
        r = evaluate_events(y_true, y_pred)
        results.append(r)
        by_cohort.setdefault(subj.split("_")[0], []).append(r)
        
summarize(results)                                          

   model_name    norm    val_f1   test_f1  n_train_subjects  n_val_subjects  \
0         cnn  window  0.872717  0.853357               166              42   
1         cnn  window  0.872717  0.853357               166              42   
2         cnn  window  0.872717  0.853357               166              42   
3         cnn  window  0.872717  0.853357               166              42   
4         cnn  window  0.872717  0.853357               166              42   
5         cnn  window  0.872717  0.853357               166              42   
6         cnn  window  0.872717  0.853357               166              42   
7         cnn  window  0.872717  0.853357               166              42   
8         cnn  window  0.872717  0.853357               166              42   
9         cnn  window  0.872717  0.853357               166              42   
10        cnn  window  0.872717  0.853357               166              42   
11        cnn  window  0.872717  0.853357           

TO: F1=0.739 P=0.592 R=0.981 | error mediano 20 ms [IQR 10-30]
HS: F1=0.734 P=0.589 R=0.975 | error mediano 20 ms [IQR 10-40]
